In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# ==============================================================================
# 1. LOAD & CLEAN DATA (Day-1 Baseline Only, Zero Leakage)
# ==============================================================================
file_path = (
    r"C:\Users\jrdie\OneDrive\Desktop\BTT\kidney_waitlist_analytic_opt.csv"
)
df = pd.read_csv(file_path, low_memory=False, encoding="latin1")

# Remove negative/missing follow-up anomalies
df = df[df["days_to_event"] > 0].copy()

# Binary Target: Adverse exit (died or removed too sick)
y = df["event_adverse"].astype(int)

# Feature Engineering from Dates (Listing time only)
init_dt = pd.to_datetime(df["INIT_DATE"], errors="coerce")
dialysis_dt = pd.to_datetime(df["DIALYSIS_DATE"], errors="coerce")
df["DIALYSIS_VINTAGE_DAYS"] = (
    (init_dt - dialysis_dt).dt.days.clip(lower=0).fillna(0)
)
df["LISTED_POST_2021"] = (init_dt >= pd.to_datetime("2021-03-15")).astype(int)

# Clean simple continuous fields
df["BMI_TCR"] = df["BMI_TCR"].clip(lower=12.0, upper=60.0)
df["BMI_TCR"] = df["BMI_TCR"].fillna(df["BMI_TCR"].median())
df["INIT_CPRA"] = df["INIT_CPRA"].fillna(0.0)
df["A2A2B_ELIGIBILITY"] = df["A2A2B_ELIGIBILITY"].fillna(-1).astype(str)

# Whitelist safe Day-1 features (Purging all post-transplant/outcome fields)
baseline_features = [
    "ON_DIALYSIS",
    "A2A2B_ELIGIBILITY",
    "GENDER",
    "ABO",
    "BMI_TCR",
    "INIT_STAT",
    "INIT_CPRA",
    "INIT_AGE",
    "ETHCAT",
    "REGION",
    "DIALYSIS_VINTAGE_DAYS",
    "LISTED_POST_2021",
]
X = df[baseline_features].copy()

# One-hot encode categorical features
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# ==============================================================================
# 2. TRAIN / TEST SPLIT
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training instances: {len(X_train):,}")
print(f"Testing instances:  {len(X_test):,}")
print(f"Baseline Adverse Prevalence: {y.mean():.4f}\n")


# Helper function to print evaluation metrics and decile lift
def evaluate_model(name, model, X_t, y_t):
  probs = model.predict_proba(X_t)[:, 1]
  preds = (probs >= 0.5).astype(int)

  roc = roc_auc_score(y_t, probs)
  pr = average_precision_score(y_t, probs)
  brier = brier_score_loss(y_t, probs)

  # Calculate Top Decile Lift
  eval_df = pd.DataFrame({"y_true": y_t.values, "prob": probs})
  eval_df["decile"] = pd.qcut(
      eval_df["prob"].rank(method="first"), q=10, labels=False
  )
  top_rate = eval_df[eval_df["decile"] == 9]["y_true"].mean()
  lift = top_rate / y_t.mean()

  print(f"=== {name} BASELINE PERFORMANCE ===")
  print(f"ROC-AUC:         {roc:.4f}  (Benchmark: >= 0.78)")
  print(f"PR-AUC:          {pr:.4f}  (Benchmark: >= 0.65)")
  print(f"Brier Score:     {brier:.4f}  (Benchmark: <= 0.18)")
  print(f"Top Decile Lift: {lift:.2f}x  (Benchmark: >= 2.50x)\n")
  return probs


# ==============================================================================
# 3. RANDOM FOREST BASELINE
# ==============================================================================
print("Training Random Forest baseline...")
# Use n_jobs=-1 and a balanced class weight to handle the ~14% positive imbalance
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_baseline.fit(X_train, y_train)
rf_probs = evaluate_model("Random Forest", rf_baseline, X_test, y_test)

# ==============================================================================
# 4. XGBOOST BASELINE
# ==============================================================================
print("Training XGBoost baseline...")
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)
xgb_baseline.fit(X_train, y_train)
xgb_probs = evaluate_model("XGBoost", xgb_baseline, X_test, y_test)

Training instances: 395,210
Testing instances:  98,803
Baseline Adverse Prevalence: 0.1409

Training Random Forest baseline...
=== Random Forest BASELINE PERFORMANCE ===
ROC-AUC:         0.7234  (Benchmark: >= 0.78)
PR-AUC:          0.2917  (Benchmark: >= 0.65)
Brier Score:     0.2061  (Benchmark: <= 0.18)
Top Decile Lift: 2.51x  (Benchmark: >= 2.50x)

Training XGBoost baseline...
=== XGBoost BASELINE PERFORMANCE ===
ROC-AUC:         0.7279  (Benchmark: >= 0.78)
PR-AUC:          0.2973  (Benchmark: >= 0.65)
Brier Score:     0.2123  (Benchmark: <= 0.18)
Top Decile Lift: 2.55x  (Benchmark: >= 2.50x)



In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# ==============================================================================
# 0. CONFIGURATION: SET LANDMARK HORIZON (e.g., 1000 Days, 1200 Days, etc.)
# ==============================================================================
LANDMARK_DAYS = 1300.0  # Adjust as needed (e.g., 1000, 1100, 1200)

# ==============================================================================
# 1. LOAD DATA & CLEAN DURATION ANOMALIES
# ==============================================================================
file_path = (
    r"C:\Users\jrdie\OneDrive\Desktop\BTT\kidney_waitlist_analytic_opt.csv"
)
df = pd.read_csv(file_path, low_memory=False, encoding="latin1")

# Remove negative/missing follow-up anomalies
df = df[df["days_to_event"] > 0].copy()

# ==============================================================================
# 2. LANDMARK COHORT & BINARY TARGET FORMULATION
# ==============================================================================
# Include candidates who either reached LANDMARK_DAYS without an adverse exit
# OR experienced an adverse exit within LANDMARK_DAYS
landmark_mask = (df["days_to_event"] >= LANDMARK_DAYS) | (
    df["event_adverse"] == 1
)
df_cohort = df[landmark_mask].copy()

# y = 1 if adverse exit within the landmark window, 0 otherwise
y = (
    (df_cohort["event_adverse"] == 1)
    & (df_cohort["days_to_event"] <= LANDMARK_DAYS)
).astype(int)

# ==============================================================================
# 3. DAY-1 BASELINE FEATURE ENGINEERING
# ==============================================================================
# Clinical date engineering
init_dt = pd.to_datetime(df_cohort["INIT_DATE"], errors="coerce")
dialysis_dt = pd.to_datetime(df_cohort["DIALYSIS_DATE"], errors="coerce")
df_cohort["DIALYSIS_VINTAGE_DAYS"] = (
    (init_dt - dialysis_dt).dt.days.clip(lower=0).fillna(0)
)
df_cohort["LISTED_POST_2021"] = (
    init_dt >= pd.to_datetime("2021-03-15")
).astype(int)

# Clean continuous fields
df_cohort["BMI_TCR"] = df_cohort["BMI_TCR"].clip(lower=12.0, upper=60.0)
df_cohort["BMI_TCR"] = df_cohort["BMI_TCR"].fillna(
    df_cohort["BMI_TCR"].median()
)
df_cohort["INIT_CPRA"] = df_cohort["INIT_CPRA"].fillna(0.0)
df_cohort["A2A2B_ELIGIBILITY"] = (
    df_cohort["A2A2B_ELIGIBILITY"].fillna(-1).astype(str)
)

# Day-1 Baseline Whitelist (No post-listing / outcome leakage)
baseline_features = [
    "ON_DIALYSIS",
    "A2A2B_ELIGIBILITY",
    "GENDER",
    "ABO",
    "BMI_TCR",
    "INIT_STAT",
    "INIT_CPRA",
    "INIT_AGE",
    "ETHCAT",
    "REGION",
    "DIALYSIS_VINTAGE_DAYS",
    "LISTED_POST_2021",
]
X = df_cohort[baseline_features].copy()

# One-hot encode categorical features
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# ==============================================================================
# 4. TRAIN / TEST SPLIT (80% Train, 20% Held-Out Test)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print(f"=== LANDMARK HORIZON: {int(LANDMARK_DAYS)} DAYS ===")
print(f"Cohort Size:        {len(df_cohort):,} patients")
print(f"Adverse Base Rate:  {y.mean():.4f} ({y.mean()*100:.2f}%)")
print(f"Train Set:          {len(X_train):,} rows")
print(f"Test Set:           {len(X_test):,} rows\n")


# Evaluation helper function
def evaluate_model(name, model, X_t, y_t):
  probs = model.predict_proba(X_t)[:, 1]

  roc = roc_auc_score(y_t, probs)
  pr = average_precision_score(y_t, probs)
  brier = brier_score_loss(y_t, probs)

  # Top Decile Lift
  eval_df = pd.DataFrame({"y_true": y_t.values, "prob": probs})
  eval_df["decile"] = pd.qcut(
      eval_df["prob"].rank(method="first"), q=10, labels=False
  )
  top_rate = eval_df[eval_df["decile"] == 9]["y_true"].mean()
  lift = top_rate / y_t.mean()

  print(f"--- {name} Baseline ({int(LANDMARK_DAYS)} Days) ---")
  print(f"ROC-AUC:         {roc:.4f}  (Target: >= 0.78)")
  print(f"PR-AUC:          {pr:.4f}  (Target: >= 0.65)")
  print(f"Brier Score:     {brier:.4f}  (Target: <= 0.18)")
  print(f"Top Decile Lift: {lift:.2f}x  (Target: >= 2.50x)\n")
  return probs


# ==============================================================================
# 5. RANDOM FOREST BASELINE
# ==============================================================================
print("Training Random Forest...")
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_baseline.fit(X_train, y_train)
rf_probs = evaluate_model("Random Forest", rf_baseline, X_test, y_test)

# ==============================================================================
# 6. XGBOOST BASELINE
# ==============================================================================
print("Training XGBoost...")
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=np.sqrt(pos_weight),  # Balances precision and recall
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)
xgb_baseline.fit(X_train, y_train)
xgb_probs = evaluate_model("XGBoost", xgb_baseline, X_test, y_test)

=== LANDMARK HORIZON: 1300 DAYS ===
Cohort Size:        129,384 patients
Adverse Base Rate:  0.4057 (40.57%)
Train Set:          103,507 rows
Test Set:           25,877 rows

Training Random Forest...
--- Random Forest Baseline (1300 Days) ---
ROC-AUC:         0.7586  (Target: >= 0.78)
PR-AUC:          0.6851  (Target: >= 0.65)
Brier Score:     0.1994  (Target: <= 0.18)
Top Decile Lift: 2.01x  (Target: >= 2.50x)

Training XGBoost...
--- XGBoost Baseline (1300 Days) ---
ROC-AUC:         0.7649  (Target: >= 0.78)
PR-AUC:          0.6950  (Target: >= 0.65)
Brier Score:     0.1923  (Target: <= 0.18)
Top Decile Lift: 2.05x  (Target: >= 2.50x)

